# Alveolar fibrosis — true 3D prototype

This notebook runs `simulations.alveolar3d`, the genuinely three-dimensional
model: alveolar centres, epithelial positions and surface normals, fibroblast
migration and orientation, neighbour searches and respiratory deformation all
live in 3D — not a display-only z coordinate.

It is a deliberately small mechanistic prototype — one central alveolus with six
near-touching neighbours across a thin interstitium — not a calibrated clinical
predictor. For the validated 2D model and its full diagnostics, use
`ipf_simulation_colab.ipynb` instead.

In [ ]:
#@title 1 · Setup { display-mode: "form" }
import os, subprocess, sys
REPO = "https://github.com/Danpc11/lung-nematic.git"  #@param {type:"string"}
ROOT = "/content/lung-nematic"
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--depth", "1", REPO, ROOT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                f"{ROOT}[simulation]"], check=True)
for name in [m for m in list(sys.modules) if m.startswith("simulations")]:
    del sys.modules[name]
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from simulations.alveolar3d import (
    Alveolar3DConfig, Alveolar3DSimulation, Alveolar3DRenderConfig,
    accelerated_3d_demo_config, human_chronic_3d_config, run_and_record_3d,
)
print("ready — true-3D alveolar model loaded")

## Choose a scenario

The accelerated demo compresses the dynamics so a run finishes in minutes. The human-chronic configuration uses the calibrated slow clock; it is far heavier and meant for offline runs.

In [ ]:
#@title 2 · Scenario { display-mode: "form" }
scenario = "accelerated demo"  #@param ["accelerated demo", "human chronic (slow, heavy)"]
total_time_days = 40  #@param {type:"slider", min:5, max:120, step:5}
seed = 12  #@param {type:"integer"}

from dataclasses import replace
if scenario.startswith("accelerated"):
    config = accelerated_3d_demo_config(seed=seed)
else:
    config = human_chronic_3d_config()
config = replace(config, total_time_h=float(total_time_days) * 24.0)
print(f"{scenario}: {total_time_days} days, dt = {config.dt_h} h")

In [ ]:
#@title 3 · Run and record { display-mode: "form" }
state_every_hours = 24  #@param {type:"slider", min:6, max:72, step:6}
make_mp4 = True  #@param {type:"boolean"}
make_gif = True  #@param {type:"boolean"}

import shutil
out_dir = "/content/results_3d"
if os.path.isdir(out_dir):
    shutil.rmtree(out_dir)
os.makedirs(out_dir, exist_ok=True)

def _progress(step):
    if step % 20 == 0:
        print(f"  step {step}")

outputs = run_and_record_3d(
    config, out_dir, state_every_h=float(state_every_hours),
    make_mp4=make_mp4, make_gif=make_gif, progress=_progress,
)
sim = outputs["simulation"]
print("done. final metrics:")
final = sim.metrics()
for key in ("time_d", "n_open", "n_collapsed", "n_indurated",
            "frac_AT1", "n_mesenchymal"):
    if key in final:
        print(f"  {key}: {final[key]}")

In [ ]:
#@title 4 · Preview the animation { display-mode: "form" }
from IPython.display import Image as _Image, Video, display
if outputs.get("gif"):
    display(_Image(outputs["gif"]))
elif outputs.get("mp4"):
    display(Video(outputs["mp4"], embed=True))
else:
    print("no animation was produced; enable make_gif or make_mp4 above")

In [ ]:
#@title 5 · Time courses { display-mode: "form" }
import pandas as pd
import matplotlib.pyplot as plt

# run_and_record_3d writes the per-step series to timeseries_3d.tsv
history = pd.read_csv(outputs["timeseries"], sep="\t")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
if "frac_AT1" in history:
    axes[0].plot(history["time_d"], history["frac_AT1"], label="AT1")
if "frac_aberrant" in history:
    axes[0].plot(history["time_d"], history["frac_aberrant"], label="aberrant")
axes[0].set_xlabel("time (days)"); axes[0].set_ylabel("epithelial fraction")
axes[0].legend(); axes[0].set_title("epithelial state")
for col, lab in [("n_open", "open"), ("n_collapsed", "collapsed"),
                 ("n_indurated", "indurated")]:
    if col in history:
        axes[1].plot(history["time_d"], history[col], label=lab)
axes[1].set_xlabel("time (days)"); axes[1].set_ylabel("alveoli")
axes[1].legend(); axes[1].set_title("alveolar state")
fig.tight_layout(); plt.show()

In [ ]:
#@title 6 · Download the run { display-mode: "form" }
import shutil
from google.colab import files
archive = shutil.make_archive("/content/alveolar3d_run", "zip", out_dir)
files.download(archive)

## Notes

The third coordinate here is physical, not a visual embedding — contrast the
2.5D `particle_render` in the 2D model, where the dynamics stay two-dimensional.
Because this model resolves real 3D neighbourhoods, it is memory-bound: keep the
alveolus count small. Seven near-touching units are enough to exercise
interdependence, loss of ventilation and focus formation without a full acinus.

This is a research prototype. The parameters are seeded from human morphometry
but the model is not calibrated against a clinical endpoint; treat its outputs as
mechanistic illustrations, not predictions.